# Lesson 14: The Agentic Loop

This notebook demonstrates building an agentic loop around an LLM and local tools. 
We move from a single-shot RAG pipeline to an iterative agent capable of multi-step search, query correction, and scope enforcement.

## 1. Setup & Baseline RAG Limitation

A standard single-shot RAG pipeline executes search once. If the user input contains a typo (e.g., `Olama` instead of `Ollama`), lexical search fails, and the pipeline cannot recover.

In [1]:
import json
from dotenv import load_dotenv
from openai import OpenAI
from ingest import load_faq_data, build_index
from rag_helper import RAGBase

load_dotenv()
openai_client = OpenAI()

# Load FAQ data & create BM25 / Elastic index
documents = load_faq_data()
index = build_index(documents)

# Initial fixed RAG setup
instructions_base = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions_base,
)

# Demonstrating failure: Lexical search fails on typos ("Olama" vs "Ollama")
answer = assistant.rag('How do I run Olama locally?')
print(answer)

The FAQ doesn’t mention **“Olama”** specifically.

If you mean running the course locally, the FAQ says you can do that instead of Codespaces if you’re comfortable setting up the needed tools: **Python, `uv`, Jupyter, Docker, and any other tools needed for the module**. It also says to **document your setup** and **keep your environment reproducible**.

If you meant something else by “Olama,” tell me the exact tool name and I’ll check the FAQ context.


## 2. Tool Definition & Function Call Dispatcher

We define our underlying `search` function, specify its JSON Schema for the API, and build `make_call` to format outputs into the response format expected by OpenAI.

In [2]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

def make_call(call):
    """
    Parses LLM tool arguments, executes the local search function,
    and returns structured function_call_output.
    """
    args = json.loads(call.arguments)

    if call.name == "search":
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

## 3. Manual Function-Calling Step

Before creating a loop, we trace a single tool-execution exchange manually to see how messages and tool outputs update conversation history.

In [3]:
question = "I just discovered the course. Can I join it?"

messages = [
    {"role": "user", "content": question}
]

# Turn 1: Model generates a function call request
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

# Append model's tool request
call = response.output[0]
messages.append(call)

# Execute tool locally & append tool output
call_output = make_call(call)
messages.append(call_output)

# Turn 2: Re-send history so the LLM reads tool results and generates a final answer
second_response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

print("Final Output:\n", second_response.output_text)

Final Output:
 Yes — you can still join and start learning anytime.

If your goal is a certificate, though, you’ll need to submit your project while the course is still accepting submissions.


## 4. The Agentic Loop Function

We wrap tool handling in a `while` loop. The loop continues making API round-trips until the model responds without requesting further function calls (`has_function_calls == False`).

In [4]:
def agent_loop(instructions: str, question: str, model: str = "gpt-5.4-mini") -> str:
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question}
    ]

    it = 1
    last_answer = ""

    while True:
        print(f"iteration #{it}...")
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == "function_call":
                print("function_call:", item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == "message":
                print("ASSISTANT:")
                last_answer = item.content[0].text
                print(last_answer)

        it += 1
        if not has_function_calls:
            break

    return last_answer

## 5. Steering Behavior & Off-Topic Guardrails

By adjusting developer instructions, we push the agent to perform multiple search queries and block off-topic requests that cannot be answered using the FAQ database.

In [5]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

# Example 1: Agent recovers from typo ('Olama' -> 'Ollama') over multiple iterations
print("--- Test 1: Typo Recovery ---")
agent_loop(instructions, "How do I run Olama locally?")

print("\n--- Test 2: Off-topic Rejection ---")
# Example 2: Agent rejects off-topic query based on FAQ constraints
agent_loop(instructions, "what's queen gambit?")

--- Test 1: Typo Recovery ---
iteration #1...
function_call: search {"query":"Olama run locally Ollama local install run locally"}
iteration #2...
function_call: search {"query":"Ollama serve localhost 11434 local run llama3 FAQ"}
iteration #3...
ASSISTANT:
To run **Ollama locally**, the course FAQ says:

1. Install Ollama from: https://ollama.com/download  
   - **macOS**: download the `.pkg`
   - **Windows**: download the `.msi`
   - **Linux**: run:
   ```bash
   curl -fsSL https://ollama.com/install.sh | sh
   ```

2. After installing, start a local model:
```bash
ollama run llama3
```

This will download the model, start it locally, and open a chat-like interface.

3. To check that the local server is running:
```bash
curl http://localhost:11434
```

4. If you want to use it from Python:
```bash
pip install ollama
```

```python
import ollama

response = ollama.chat(
    model='llama3',
    messages=[{"role": "user", "content": your_prompt}]
)

print(response['message']['content'])

'I couldn’t find anything in the course FAQ about “queen’s gambit,” so it looks off-topic for this course.\n\nIf you meant a course-related term, feel free to ask with a bit more context. Are there other areas you want to explore?'